<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/04_week/entropy_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math

#Pure Certainty
# [EN] WHY WE WROTE THIS: To demonstrate the absolute lower bound of Shannon Entropy (0.0 Bits).
# [TR] NEDEN YAZDIK: Shannon Entropisinin mutlak alt sınırını (0.0 Bit) görmek ve formülü anlamak için.
# [EN] GOAL: Calculate entropy when a model is 100% confident in a single outcome.
# [TR] AMAÇ: Model tek bir seçenekten %100 emin olduğunda entropinin sıfırlandığını göstermek.

p_certain = [1.0, 0.0]
# Formül / Formula: - sum(P * log2(P))  (0 * log2(0) = 0 kabul edilir)
entropy_1 = -(1.0 * math.log2(1.0) + 0)

print(f"[Code 1] Tam Kesinlik Entropisi / Pure Certainty Entropy: {entropy_1:.4f}")

# [EN] RESULT INSIGHT: 0.0 Bits means absolute certainty. There is zero unpredictability or information chaos.
# [TR] SONUÇ NE SÖYLÜYOR?: 0.0 Bit tam kesinlik demektir. Hiçbir belirsizlik veya bilgi karmaşası yoktur.

[Code 1] Tam Kesinlik Entropisi / Pure Certainty Entropy: -0.0000


In [3]:
#Pure Uncertainty
# [EN] WHY WE WROTE THIS: To demonstrate maximum binary entropy (1.0 Bit) using an equal probability split.
# [TR] NEDEN YAZDIK: Eşit olasılık dağılımında maksimum ikili entropi değerini (1.0 Bit) göstermek için.
# [EN] GOAL: Measure uncertainty in a coin-flip scenario (50-50 decision split).
# [TR] AMAÇ: %50-%50 yazı-tura senaryosundaki kararsızlık miktarını ölçmek.

p_uncertain = [0.5, 0.5]
entropy_2 = -(0.5 * math.log2(0.5) + 0.5 * math.log2(0.5))

print(f"[Code 2] Tam Kararsızlık Entropisi / Pure Uncertainty Entropy: {entropy_2:.4f}")

# [EN] RESULT INSIGHT: 1.0 Bit is the maximum possible uncertainty for a 2-choice system. The model is totally guessing.
# [TR] SONUÇ NE SÖYLÜYOR?: 1.0 Bit, 2 seçenekli sistemdeki maksimum kararsızlıktır. Model tamamen rastgele tahmin yapmaktadır.

[Code 2] Tam Kararsızlık Entropisi / Pure Uncertainty Entropy: 1.0000


In [4]:
import torch

# [EN] WHY WE WROTE THIS: Math crash protection. Log2(0) is undefined and produces NaN/Infinity errors in PyTorch.
# [TR] NEDEN YAZDIK: Matematiksel çökme koruması. Log2(0) tanımsızdır ve PyTorch'ta NaN/Sonsuz hatalarına yol açar.
# [EN] GOAL: Add a tiny value (1e-10) to probabilities to make log operations numerically stable.
# [TR] AMAÇ: Olasılıklara çok küçük bir epsilon (1e-10) ekleyerek logaritma işlemini güvenli ve kararlı hale getirmek.

eps = 1e-10
prob_with_zero = torch.tensor([1.0, 0.0, 0.0])
safe_log = torch.log2(prob_with_zero + eps)
entropy_3 = -torch.sum(prob_with_zero * safe_log).item()

print(f"[Code 3] Epsilon Korumalı Entropi / Epsilon-Protected Entropy: {entropy_3:.4f}")

# [EN] RESULT INSIGHT: Safely returns 0.0 instead of crashing code or returning NaN.
# [TR] SONUÇ NE SÖYLÜYOR?: Kodun çökmesini önleyerek güvenli şekilde 0.0 sonucunu verir.

[Code 3] Epsilon Korumalı Entropi / Epsilon-Protected Entropy: -0.0000


In [5]:
# [EN] WHY WE WROTE THIS: To perform fast tensor calculations without writing manual for-loops.
# [TR] NEDEN YAZDIK: Manuel for döngüleri yazmadan hızlı tensor hesaplamaları yapmak için.
# [EN] GOAL: Calculate multi-class Shannon Entropy using PyTorch vectorized operations.
# [TR] AMAÇ: PyTorch'un vektörel işlemlerini kullanarak çok sınıflı Shannon Entropisini tek satırda hesaplamak.

probs_tensor = torch.tensor([0.70, 0.20, 0.05, 0.05])
entropy_4 = -torch.sum(probs_tensor * torch.log2(probs_tensor + 1e-10)).item()

print(f"[Code 4] Vektörel Tensor Entropisi / Vectorized Tensor Entropy: {entropy_4:.4f}")

# [EN] RESULT INSIGHT: Entropy is 1.18. The model heavily leans on the first option (70%), keeping uncertainty relatively low.
# [TR] SONUÇ NE SÖYLÜYOR?: Entropi 1.18 çıktı. Model %70 ile ilk seçeneğe ağırlık verdiği için kararsızlık göreceli olarak düşüktür.

[Code 4] Vektörel Tensor Entropisi / Vectorized Tensor Entropy: 1.2568


In [6]:
import torch.nn.functional as F

# [EN] WHY WE WROTE THIS: LLMs produce unnormalized raw scores (logits). We must convert them to percentages.
# [TR] NEDEN YAZDIK: LLM'ler ölçeklenmemiş ham skorlar (logitler) üretir. Bunları yüzdelik oranlara çevirmeliyiz.
# [EN] GOAL: Map raw logit values into a valid probability distribution that sums up to 1.0 (100%).
# [TR] AMAÇ: Ham logit değerlerini toplamı 1.0 (%100) eden geçerli bir olasılık dağılımına dönüştürmek.

sample_logits = torch.tensor([12.5, 3.1, -1.2, 0.5])
sample_probs = F.softmax(sample_logits, dim=-1)

print(f"[Code 5] Softmax Olasılıkları / Softmax Probabilities: {sample_probs.tolist()}")

# [EN] RESULT INSIGHT: The score 12.5 exponentially dominates the rest, receiving almost ~99.9% of the probability weight.
# [TR] SONUÇ NE SÖYLÜYOR?: 12.5 skoru diğerlerini ezerek olasılık ağırlığının neredeyse %99.9'unu tek başına almıştır.

[Code 5] Softmax Olasılıkları / Softmax Probabilities: [0.9999099969863892, 8.271665137726814e-05, 1.1223454521314125e-06, 6.1436594478436746e-06]


In [7]:
# [EN] WHY WE WROTE THIS: To build a reusable helper function that takes raw model outputs and directly returns entropy.
# [TR] NEDEN YAZDIK: Ham model çıktılarını alıp doğrudan entropi döndüren tekrar kullanılabilir bir yardımcı fonksiyon yazmak için.
# [EN] GOAL: Combine Softmax transformation and Shannon Entropy calculation into a single modular function.
# [TR] AMAÇ: Softmax dönüşümünü ve Shannon Entropisi hesaplamasını modüler tek bir fonksiyonda birleştirmek.

def logits_to_entropy(logits_vec):
    p = F.softmax(logits_vec, dim=-1)
    return -torch.sum(p * torch.log2(p + 1e-10)).item()

entropy_6 = logits_to_entropy(sample_logits)
print(f"[Code 6] Logit'ten Hesaplanan Entropi / Entropy from Logits: {entropy_6:.4f}")

# [EN] RESULT INSIGHT: Very low entropy (~0.008) confirms the model is extremely certain about its top prediction.
# [TR] SONUÇ NE SÖYLÜYOR?: Çok düşük entropi (~0.008), modelin ilk tahmini konusunda son derece kendinden emin olduğunu doğrular.

[Code 6] Logit'ten Hesaplanan Entropi / Entropy from Logits: 0.0014


In [8]:
# [EN] WHY WE WROTE THIS: To contrast a confident output distribution against a highly hesitant one.
# [TR] NEDEN YAZDIK: Kendinden emin bir çıktı dağılımı ile kararsız/tereddütlü bir dağılımı kıyaslamak için.
# [EN] GOAL: Prove that spread-out logits result in significantly higher entropy values.
# [TR] AMAÇ: Birbirine yakın (yayılmış) logitlerin çok daha yüksek entropi ürettiğini kanıtlamak.

high_conf_logits = torch.tensor([15.0, 1.0, 0.0, -2.0])
low_conf_logits = torch.tensor([4.0, 3.9, 3.8, 3.7])

print(f"[Code 7] Yüksek Güven Entropisi / High Conf Entropy: {logits_to_entropy(high_conf_logits):.4f}")
print(f"[Code 7] Düşük Güven Entropisi / Low Conf Entropy  : {logits_to_entropy(low_conf_logits):.4f}")

# [EN] RESULT INSIGHT: Confident logits yield near-zero entropy (0.0001), while hesitant logits yield high entropy (1.9992).
# [TR] SONUÇ NE SÖYLÜYOR?: Emin logitler sıfıra yakın (0.0001), kararsız logitler ise yüksek entropi (1.9992) verir.

[Code 7] Yüksek Güven Entropisi / High Conf Entropy: 0.0000
[Code 7] Düşük Güven Entropisi / Low Conf Entropy  : 1.9910
